# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [7]:
# Using Paper data to reproduce the results
context = [ {'role':'system', 'content':"""
-- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.

 create table highschooler (
id int primary key ,
name text ,
grade int
);
             
create table friend (
student_id int ,
friend_id int ,
primary key ( student_id , friend_id ) ,
foreign key ( student_id ) references
highschooler ( id ) ,
foreign key ( friend_id ) references
highschooler ( id )
);
             
create table likes (
student_id int ,
liked_id int ,
primary key ( student_id , liked_id ) ,
foreign key ( liked_id ) references
highschooler ( id ) ,
foreign key ( student_id ) references
highschooler ( id )
);

/*             
Example1:
User question: What is the name of the highest grade in highscooler table:
SQL: SELECT name FROM highscooler WHERE grade = (SELECT MAX(grade) FROM highscooler) LIMIT 1;
*/
             
/*    
Example2:
User question: Which are the friends of a student?
SQL: SELECT highschooler.student_id, friend.liked_id FROM highschooler JOIN friend ON highschooler.id = friend.student_id
*/           
             
/* 
Example3: What does a student like?
User question: SELECT highschooler.student_id, likes.liked_id FROM highschooler JOIN likes ON highschooler.id = likes.student_id
*/
"""} ]

In [4]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
"""
})

In [8]:
#Function to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [14]:
#new
context_user = context.copy()
print(return_CCRMSQL("""Give me the mean grade of all students""", context_user))

SELECT AVG(grade) AS mean_grade FROM highschooler;


In [11]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Give me the 2 highest salary", old_context_user))

This is your SQL:
```sql
SELECT *
FROM salary
ORDER BY salary DESC
LIMIT 2;
```

This SQL query selects all columns from the `salary` table, orders the results by the `salary` column in descending order, and limits the output to the top 2 highest salaries.


In [ ]:
#new
context_user = context.copy()
print(return_CCRMSQL("Return the likes of a student order by their grade", context_user))

SELECT h.name, l.liked_id
FROM highschooler h
JOIN likes l ON h.id = l.student_id
ORDER BY h.grade;


In [ ]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("What's the count of workers?", old_context_user))

This is your SQL:
```sql
SELECT COUNT(*) AS worker_count FROM employees;
```

Explanation: This SQL query selects the count of all rows in the "employees" table, which represents the number of workers in the database.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [16]:
#new
context_user = context.copy()
print(return_CCRMSQL("""Give me the average grade of students table""", context_user))

SELECT AVG(grade) FROM highschooler;


In [ ]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("What's the average wage overall?", old_context_user))

This is your SQL:
```sql
SELECT AVG(salary) AS average_wage
FROM salary;
```

This SQL query calculates the average salary from the "salary" table, providing the overall average wage.


In [17]:
#new
context_user = context.copy()
print(return_CCRMSQL("""Return the number of liked things for every student""", context_user))

SELECT student_id, COUNT(liked_id) AS num_likes
FROM likes
GROUP BY student_id;


In [18]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("What are the studies of each employee order by year?", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.year, st.educational_level, st.Institution, st.Years, st.Speciality
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.year;
```

This SQL query retrieves the name of each employee, their educational level, institution, years of study, and speciality, ordering the results by the year of their salary.


In [19]:
#new
context_user = context.copy()
print(return_CCRMSQL("""Return the number of loved things for every student""", context_user))

SELECT student_id, COUNT(liked_id) AS num_loved_things
FROM likes
GROUP BY student_id;


# Summary
`gpt-3.5-turbo` produced accurate results accross the tests. We tried to trick the model with synonyms like `wage` instead of salary but it reasoned well and the output was correct.

A new schema few-shots context was created to try it in different situations, but in this case the model did not hallucinated. It understands the underlying sense of SQL output.

The simplicity of the new schema might have helped to get this good results. In any case evrything seems to work flawlessly in this short Lab.

## What I learn
* Few-shot prompting is strongly linked to the output structure
* It's interesting how well it performed in old LLMs